In [1]:
import random
import time
import numpy as np
from enum import Enum
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque

class Direction(Enum):
    NORTH = 0
    SOUTH = 1
    EAST  = 2
    WEST  = 3

MOVES = {
    Direction.NORTH: (-1, 0),
    Direction.SOUTH: (1, 0),
    Direction.EAST:  (0, 1),
    Direction.WEST:  (0, -1),
}

GRID_SIZE = 5
NUM_AGENTS = 4

class GridEnvironment:
    def __init__(self):
        self.grid_size = GRID_SIZE
        self.num_agents = NUM_AGENTS
        self.reset()

    def reset(self, A=None, B=None):
        if A is not None and B is not None:
            self.A = A
            self.B = B
        else:
            self.A = (random.randint(0, GRID_SIZE-1), random.randint(0, GRID_SIZE-1))
            self.B = (random.randint(0, GRID_SIZE-1), random.randint(0, GRID_SIZE-1))
            while self.A == self.B:
                self.B = (random.randint(0, GRID_SIZE-1), random.randint(0, GRID_SIZE-1))

        self.agents = []
        for _ in range(self.num_agents):
            self.agents.append({
                'row': self.A[0],
                'col': self.A[1],
                'has_item': True
            })

        self.step_count = 0
        self.collision_count = 0
        return self._get_states()

    def _get_states(self):
        states = []
        for ag in self.agents:
            r, c = ag['row'], ag['col']
            s = np.array([
                r / 4.0,
                c / 4.0,
                1.0 if ag['has_item'] else 0.0,
                self.A[0] / 4.0,
                self.A[1] / 4.0,
                self.B[0] / 4.0,
                self.B[1] / 4.0,
                (r - self.A[0]) / 4.0,
                (c - self.A[1]) / 4.0,
                (r - self.B[0]) / 4.0,
                (c - self.B[1]) / 4.0,
            ], dtype=np.float32)
            states.append(s)
        return states

    def _apply_action(self, agent_idx, action):
        agent = self.agents[agent_idx]
        dr, dc = MOVES[action]
        old_r, old_c = agent['row'], agent['col']
        new_r = max(0, min(GRID_SIZE - 1, old_r + dr))
        new_c = max(0, min(GRID_SIZE - 1, old_c + dc))
        agent['row'] = new_r
        agent['col'] = new_c
        return (new_r == old_r and new_c == old_c) # wall bump

    def _detect_collisions(self):
        collided_indices = set()
        pos_map = {}
        for i, ag in enumerate(self.agents):
            pos = (ag['row'], ag['col'])
            pos_map.setdefault(pos, []).append(i)

        for pos, indices in pos_map.items():
            if pos == self.A or pos == self.B:
                continue
            if len(indices) > 1:
                has_items = [self.agents[i]['has_item'] for i in indices]
                if True in has_items and False in has_items:
                    collided_indices.update(indices)

        return list(collided_indices)

    def step(self, actions):
        order = list(range(self.num_agents))
        random.shuffle(order)

        wall_bumps = [False] * self.num_agents
        for idx in order:
            wall_bumps[idx] = self._apply_action(idx, actions[idx])

            ag = self.agents[idx]
            if (ag['row'], ag['col']) == self.A and not ag['has_item']:
                ag['has_item'] = True
            elif (ag['row'], ag['col']) == self.B and ag['has_item']:
                ag['has_item'] = False

        collided = self._detect_collisions()
        self.collision_count += len(collided)

        rewards = [0.0] * self.num_agents
        for i, ag in enumerate(self.agents):
            r = -0.5
            if wall_bumps[i]:
                r -= 1.0

            if (ag['row'], ag['col']) == self.B and not ag['has_item']:
                r += 20.0
            elif (ag['row'], ag['col']) == self.A and ag['has_item']:
                r += 20.0

            if i in collided:
                r -= 50.0

            rewards[i] = r

        self.step_count += 1
        next_states = self._get_states()
        return next_states, rewards, False

class DQN(nn.Module):
    def __init__(self, input_dim=11, output_dim=4):
        super(DQN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.fc(x)

class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size=64):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)),
                torch.LongTensor(a),
                torch.FloatTensor(r),
                torch.FloatTensor(np.array(ns)),
                torch.FloatTensor(d))

    def __len__(self):
        return len(self.buffer)

def train_dqn():
    env = GridEnvironment()
    q_net = DQN()
    target_net = DQN()
    target_net.load_state_dict(q_net.state_dict())
    optimizer = optim.Adam(q_net.parameters(), lr=1e-3)
    buffer = ReplayBuffer()

    epsilon = 1.0
    epsilon_min = 0.02
    epsilon_decay = 0.9995

    batch_size = 64
    gamma = 0.99
    step_budget = 0
    max_steps = 1_5000
    start_time = time.time()

    states = env.reset()
    ep_len = 0

    print("Training PyTorch DQN...")

    while step_budget < max_steps and env.collision_count < 4000:
        if time.time() - start_time > 600:
            break

        actions = []
        for i in range(NUM_AGENTS):
            if random.random() < epsilon:
                act = random.randint(0, 3)
            else:
                with torch.no_grad():
                    st_tensor = torch.FloatTensor(states[i]).unsqueeze(0)
                    q_vals = q_net(st_tensor)
                    act = torch.argmax(q_vals, dim=1).item()
            actions.append(Direction(act))

        next_states, rewards, _ = env.step(actions)
        step_budget += 4
        ep_len += 1

        for i in range(NUM_AGENTS):
            buffer.push(states[i], actions[i].value, rewards[i], next_states[i], False)

        if len(buffer) >= batch_size:
            s_b, a_b, r_b, ns_b, d_b = buffer.sample(batch_size)
            q_val = q_net(s_b).gather(1, a_b.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                max_ns_q = target_net(ns_b).max(1)[0]
                target_q = r_b + gamma * max_ns_q * (1 - d_b)

            loss = nn.MSELoss()(q_val, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if step_budget % 1000 == 0:
            target_net.load_state_dict(q_net.state_dict())

        states = next_states
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

        # Check if all agents completed round trip or max steps reached
        if ep_len >= 40:
            states = env.reset()
            ep_len = 0

        if step_budget % 100_000 == 0:
            elapsed = time.time() - start_time
            print(f"Agent Steps: {step_budget:8d} | Collisions: {env.collision_count:4d} | Epsilon: {epsilon:.3f} | Time: {elapsed:.1f}s")

    elapsed = time.time() - start_time
    print(f"\nDQN Training Complete. Steps: {step_budget}, Collisions: {env.collision_count}, Time: {elapsed:.2f}s")
    return q_net, env

def test_dqn_performance(q_net, num_scenarios=1000):
    env = GridEnvironment()
    q_net.eval()
    successes = 0

    print(f"\nEvaluating DQN performance over {num_scenarios} scenarios...")

    for sc in range(num_scenarios):
        states = env.reset()
        reached_B = [False] * NUM_AGENTS
        returned_A = [False] * NUM_AGENTS

        for step in range(25):
            actions = []
            for i in range(NUM_AGENTS):
                with torch.no_grad():
                    st_tensor = torch.FloatTensor(states[i]).unsqueeze(0)
                    act = torch.argmax(q_net(st_tensor), dim=1).item()
                actions.append(Direction(act))

            next_states, rewards, _ = env.step(actions)

            if env.collision_count > 0:
                break

            for i, ag in enumerate(env.agents):
                if (ag['row'], ag['col']) == env.B and not ag['has_item']:
                    reached_B[i] = True
                elif reached_B[i] and (ag['row'], ag['col']) == env.A and ag['has_item']:
                    returned_A[i] = True

            if all(returned_A):
                successes += 1
                break

            states = next_states

    success_rate = (successes / num_scenarios) * 100
    print(f"DQN Success Rate: {success_rate:.2f}% ({successes}/{num_scenarios})")
    return success_rate

if __name__ == "__main__":
    q_net, env = train_dqn()
    test_dqn_performance(q_net)

Training PyTorch DQN...

DQN Training Complete. Steps: 15000, Collisions: 0, Time: 8.14s

Evaluating DQN performance over 1000 scenarios...
DQN Success Rate: 0.50% (5/1000)
